In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import eye as speye, lil_matrix
from itertools import combinations
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import math
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# ILP-BASED EXACT DAG STRUCTURE LEARNING
# ==========================================
# Replaces HillClimbSearch with an Integer Linear Programming
# formulation that finds the globally optimal BIC-scoring DAG.
#
# Uses scipy.optimize.milp (HiGHS solver) — no external packages needed.
#
# Approach:
#   1. Pre-compute BIC score for every candidate parent set (up to max_parents)
#   2. Create binary variables for each (node, parent_set) choice
#   3. Add constraints: exactly one parent set per node
#   4. Add acyclicity constraints via topological ordering variables
#   5. Solve to global optimum
# ==========================================


def compute_bic_score_discrete(data, child, parents):
    """
    Compute the BIC score for a child given a parent set.
    For discrete data: BIC = LL - (k/2) * ln(N)
    where LL is the log-likelihood, k is the number of free parameters,
    and N is the sample size.

    Returns the LOCAL BIC score (higher = better).
    """
    N = len(data)
    if N == 0:
        return -np.inf

    parent_list = list(parents)

    if len(parent_list) == 0:
        # No parents — just count child categories
        child_counts = data[child].value_counts()
        r_child = len(child_counts)
        ll = 0.0
        for count in child_counts:
            if count > 0:
                ll += count * np.log(count / N)
        k = r_child - 1
        return ll - (k / 2.0) * np.log(N)

    # Group by parents, count child values within each parent config
    grouped = data.groupby(parent_list)[child].value_counts()
    parent_totals = data.groupby(parent_list)[child].count()

    r_child = data[child].nunique()
    q_parents = len(parent_totals)  # number of parent configurations observed

    single_parent = (len(parent_list) == 1)

    ll = 0.0
    for idx, count in grouped.items():
        if single_parent:
            # idx is (parent_val, child_val) — flat tuple, not nested
            parent_key = idx[0]
            parent_total = parent_totals[parent_key]
        else:
            # idx is (p1_val, p2_val, ..., child_val)
            parent_key = idx[:-1]
            parent_total = parent_totals[parent_key]
        if count > 0 and parent_total > 0:
            ll += count * np.log(count / parent_total)

    k = q_parents * (r_child - 1)
    return ll - (k / 2.0) * np.log(N)


def enumerate_parent_sets(nodes, child, max_parents=3):
    """
    Enumerate all possible parent sets for a given child node,
    up to max_parents size. Returns list of frozensets.
    """
    candidates = [n for n in nodes if n != child]
    parent_sets = [frozenset()]  # empty parent set is always an option

    for size in range(1, min(max_parents, len(candidates)) + 1):
        for combo in combinations(candidates, size):
            parent_sets.append(frozenset(combo))

    return parent_sets


def precompute_all_scores(data, nodes, max_parents=3):
    """
    Pre-compute BIC scores for every (child, parent_set) combination.
    Returns:
        scores: dict of {child: [(parent_set, bic_score), ...]}
    """
    import time

    # Count total work upfront
    total_parent_sets = 0
    for child in nodes:
        ps_count = len(enumerate_parent_sets(nodes, child, max_parents))
        total_parent_sets += ps_count

    scores = {}
    completed = 0
    start_time = time.time()

    for node_idx, child in enumerate(nodes):
        node_start = time.time()
        child_scores = []
        parent_sets = enumerate_parent_sets(nodes, child, max_parents)

        for ps_idx, ps in enumerate(parent_sets):
            bic = compute_bic_score_discrete(data, child, ps)
            child_scores.append((ps, bic))
            completed += 1

            # Print progress every 50 parent sets
            if completed % 50 == 0 or completed == total_parent_sets:
                elapsed = time.time() - start_time
                rate = completed / elapsed if elapsed > 0 else 0
                remaining = (total_parent_sets - completed) / rate if rate > 0 else 0
                print(f"\r    Progress: {completed}/{total_parent_sets} parent sets "
                      f"({completed/total_parent_sets*100:.1f}%) | "
                      f"Elapsed: {elapsed:.0f}s | "
                      f"ETA: {remaining:.0f}s", end="", flush=True)

        node_elapsed = time.time() - node_start
        scores[child] = child_scores
        print(f"\n    Node {node_idx+1}/{len(nodes)}: '{child}' — "
              f"{len(parent_sets)} parent sets scored in {node_elapsed:.1f}s")

    total_elapsed = time.time() - start_time
    print(f"  BIC pre-computation complete: {total_parent_sets} scores in {total_elapsed:.1f}s")
    return scores


def solve_dag_ilp(nodes, scores, verbose=True):
    """
    Solve for the globally optimal DAG using ILP.

    Decision variables:
        w[i][k] ∈ {0,1} — whether node i uses its k-th candidate parent set
        order[i] ∈ [0, n-1] — topological position of node i (continuous)

    Objective: maximize total BIC = sum over all nodes of selected parent set's BIC score

    Constraints:
        1. For each node i: sum_k w[i][k] = 1  (exactly one parent set chosen)
        2. Acyclicity via ordering: if parent set k of node i contains node j,
           then order[j] < order[i]. Encoded as:
           order[i] - order[j] >= 1 - n*(1 - w[i][k])
           i.e.  order[i] - order[j] + n*w[i][k] >= 1 + n - n = 1
           Rearranged: order[i] - order[j] + n*w[i][k] >= 1 - n*(1)  ... 
           
           Actually the standard big-M linearization:
           If w[i][k] = 1 and parent set k contains j, then order[j] < order[i].
           order[i] - order[j] >= 1 - M*(1 - w[i][k])
           where M = n (number of nodes).
    """
    n = len(nodes)
    node_idx = {node: i for i, node in enumerate(nodes)}

    # Build variable index map
    # w variables: w[i][k] for node i, parent set index k
    w_vars = {}  # (node_index, parent_set_index) -> variable index
    var_count = 0
    for i, node in enumerate(nodes):
        for k in range(len(scores[node])):
            w_vars[(i, k)] = var_count
            var_count += 1
    n_w = var_count

    # order variables: order[i] for each node — continuous [0, n-1]
    order_start = var_count
    var_count += n
    n_total = var_count

    if verbose:
        print(f"  ILP: {n} nodes, {n_w} parent-set binary vars, {n} order vars")
        print(f"  Total variables: {n_total}")

    # === OBJECTIVE ===
    # Maximize sum of BIC scores for selected parent sets
    # scipy.milp MINIMIZES, so we negate
    c = np.zeros(n_total)
    for i, node in enumerate(nodes):
        for k, (ps, bic) in enumerate(scores[node]):
            c[w_vars[(i, k)]] = -bic  # negate for minimization

    # === VARIABLE BOUNDS ===
    # w vars: binary [0, 1] (enforced via integrality)
    # order vars: continuous [0, n-1]
    lower = np.zeros(n_total)
    upper = np.ones(n_total)
    for i in range(n):
        upper[order_start + i] = n - 1

    # === INTEGRALITY ===
    # 1 = integer (binary since bounds are 0-1), 0 = continuous
    integrality = np.zeros(n_total)
    integrality[:n_w] = 1  # w vars are binary

    # === CONSTRAINTS ===
    # We'll build constraint rows as lists, then stack

    A_rows = []
    b_lower = []
    b_upper = []

    # Constraint 1: For each node, exactly one parent set selected
    # sum_k w[i][k] = 1  for each i
    for i, node in enumerate(nodes):
        row = np.zeros(n_total)
        for k in range(len(scores[node])):
            row[w_vars[(i, k)]] = 1.0
        A_rows.append(row)
        b_lower.append(1.0)
        b_upper.append(1.0)

    # Constraint 2: Acyclicity ordering constraints
    # For each node i, parent set k, and parent j in that set:
    #   order[i] - order[j] >= 1 - M*(1 - w[i][k])
    #   order[i] - order[j] + M*w[i][k] >= 1 - M + M = 1
    #   Actually: order[i] - order[j] >= 1 - M + M*w[i][k]  ... let me redo.
    #
    #   We want: IF w[i][k]=1 THEN order[i] - order[j] >= 1
    #   Big-M: order[i] - order[j] + M*(1 - w[i][k]) >= 1
    #   => order[i] - order[j] - M*w[i][k] >= 1 - M
    M = n  # big-M constant

    for i, node in enumerate(nodes):
        for k, (ps, bic) in enumerate(scores[node]):
            for parent in ps:
                j = node_idx[parent]
                # order[i] - order[j] - M*w[i][k] >= 1 - M
                row = np.zeros(n_total)
                row[order_start + i] = 1.0
                row[order_start + j] = -1.0
                row[w_vars[(i, k)]] = -M
                A_rows.append(row)
                b_lower.append(1.0 - M)
                b_upper.append(np.inf)

    if verbose:
        print(f"  Constraints: {len(A_rows)} total")

    if len(A_rows) == 0:
        # Edge case: no constraints (shouldn't happen with >1 node)
        return []

    A = np.array(A_rows)
    constraints = LinearConstraint(A, b_lower, b_upper)
    bounds = Bounds(lower, upper)

    # === SOLVE ===
    if verbose:
        print("  Solving ILP (this may take a moment)...")

    result = milp(
        c=c,
        constraints=constraints,
        integrality=integrality,
        bounds=bounds,
        options={"disp": False, "time_limit": 300}  # 5 min timeout
    )

    if not result.success:
        print(f"  WARNING: ILP solver did not find optimal solution: {result.message}")
        return []

    optimal_bic = -result.fun
    if verbose:
        print(f"  Optimal total BIC score: {optimal_bic:.2f}")

    # === EXTRACT EDGES ===
    x = result.x
    edges = []
    for i, node in enumerate(nodes):
        for k, (ps, bic) in enumerate(scores[node]):
            if x[w_vars[(i, k)]] > 0.5:  # selected (binary, so ~1.0)
                for parent in ps:
                    edges.append((parent, node))
                if verbose and len(ps) > 0:
                    print(f"    {node} <- {set(ps)}  (BIC: {bic:.2f})")
                break

    return edges


def learn_dag_ilp(data, max_parents=3, verbose=True):
    """
    Main entry point: learn an optimal DAG from discrete data using ILP.
    Returns a list of (parent, child) edge tuples.
    """
    nodes = list(data.columns)
    if verbose:
        print(f"\nLearning optimal DAG for {len(nodes)} variables...")
        print(f"  Max parents per node: {max_parents}")
        print(f"  Sample size: {len(data)}")

    # Step 1: Pre-compute all BIC scores
    if verbose:
        print("  Pre-computing BIC scores for all candidate parent sets...")
    scores = precompute_all_scores(data, nodes, max_parents)

    total_candidates = sum(len(v) for v in scores.values())
    if verbose:
        print(f"  Total candidate parent sets: {total_candidates}")

    # Step 2: Solve ILP
    edges = solve_dag_ilp(nodes, scores, verbose=verbose)

    if verbose:
        print(f"  Learned DAG with {len(edges)} edges")

    return edges


# ==========================================
# 1. LOAD FULL DATASET
# ==========================================
print("Loading the full Patient Vault...")
df_vault = pd.read_csv("Cleaned_Patient_Vault.csv")
df_vault = df_vault.drop(columns=["RECORD_ID"])
print(f"Total rows available: {len(df_vault)}")

# ==========================================
# 2. BOOTSTRAP SETTINGS
# ==========================================
N_RUNS = 1
SAMPLE_SIZE = 706000
STABILITY_THRESHOLD = 0.80

clinical_counter = Counter()
financial_counter = Counter()

# ==========================================
# 3. BOOTSTRAP LOOP (now with ILP)
# ==========================================
print(f"\n--- Running {N_RUNS} bootstrap iterations with ILP-based DAG learning ---\n")

for i in range(N_RUNS):
    if (i + 1) % 10 == 0 or N_RUNS == 1:
        print(f"Run {i + 1}/{N_RUNS}...")

    df_sample = df_vault.sample(n=SAMPLE_SIZE, replace=True, random_state=i).copy()

    df_sample['TOTAL_CHARGES'] = pd.qcut(
        df_sample['TOTAL_CHARGES'], q=4,
        labels=['Low', 'Medium', 'High', 'Very High']
    )
    df_sample['LENGTH_OF_STAY'] = pd.qcut(
        df_sample['LENGTH_OF_STAY'], q=4,
        labels=['Short', 'Medium', 'Long', 'Extended'],
        duplicates='drop'
    )
    df_sample = df_sample.astype(str)

    try:
        # ---- ILP REPLACES HILL CLIMBING HERE ----
        edges = learn_dag_ilp(df_sample, max_parents=3, verbose=True)

        # Build a networkx DiGraph to extract Markov blankets
        G = nx.DiGraph()
        G.add_nodes_from(df_sample.columns)
        G.add_edges_from(edges)

        # Markov blanket = parents + children + co-parents (other parents of my children)
        def get_markov_blanket(G, target):
            parents = set(G.predecessors(target))
            children = set(G.successors(target))
            co_parents = set()
            for child in children:
                co_parents.update(G.predecessors(child))
            blanket = parents | children | co_parents
            blanket.discard(target)
            return blanket

        mb_clinical = get_markov_blanket(G, "LENGTH_OF_STAY")
        mb_financial = get_markov_blanket(G, "TOTAL_CHARGES")

        for var in mb_clinical:
            clinical_counter[var] += 1
        for var in mb_financial:
            financial_counter[var] += 1

    except Exception as e:
        print(f"  Run {i + 1} failed: {e}")
        import traceback
        traceback.print_exc()
        continue

# ==========================================
# 4. STABILITY REPORT
# ==========================================
all_vars = sorted([c for c in df_vault.columns if c not in ["LENGTH_OF_STAY", "TOTAL_CHARGES"]])

print("\n" + "=" * 65)
print(f"  MARKOV BLANKET STABILITY REPORT  ({N_RUNS} bootstrap runs)")
print(f"  Method: ILP-based exact DAG learning (globally optimal BIC)")
print("=" * 65)
print(f"{'Variable':<25} {'Clinical (LOS)':<18} {'Financial (TC)':<18}")
print("-" * 65)
for var in all_vars:
    c_count = clinical_counter.get(var, 0)
    f_count = financial_counter.get(var, 0)
    c_pct = f"{c_count}/{N_RUNS} ({c_count/N_RUNS*100:.0f}%)"
    f_pct = f"{f_count}/{N_RUNS} ({f_count/N_RUNS*100:.0f}%)"
    print(f"{var:<25} {c_pct:<18} {f_pct:<18}")
print("-" * 65)

# ==========================================
# 5. AUTO-EXTRACT STABLE BLANKET VARIABLES
# ==========================================
threshold_count = int(N_RUNS * STABILITY_THRESHOLD)

clinical_blanket = [var for var, count in clinical_counter.items() if count >= threshold_count]
financial_blanket = [var for var, count in financial_counter.items() if count >= threshold_count]

print(f"\nStable Clinical Blanket (80%+):  {clinical_blanket}")
print(f"Stable Financial Blanket (80%+): {financial_blanket}")

if not clinical_blanket:
    print("\nWARNING: No variables hit 80% for Clinical. Lowering threshold to 50%.")
    clinical_blanket = [var for var, count in clinical_counter.items() if count >= N_RUNS * 0.5]
if not financial_blanket:
    print("WARNING: No variables hit 80% for Financial. Lowering threshold to 50%.")
    financial_blanket = [var for var, count in financial_counter.items() if count >= N_RUNS * 0.5]

# ==========================================
# 6. STABILITY CHART
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if clinical_counter:
    vars_c = sorted(clinical_counter.keys(), key=lambda x: clinical_counter[x], reverse=True)
    counts_c = [clinical_counter[v] for v in vars_c]
    colors_c = ['#2ecc71' if c >= N_RUNS * 0.8 else '#f39c12' if c >= N_RUNS * 0.5 else '#e74c3c' for c in counts_c]
    axes[0].barh(vars_c, counts_c, color=colors_c, edgecolor='black')
    axes[0].axvline(x=N_RUNS * 0.8, color='green', linestyle='--', linewidth=2, label='80% threshold')
    axes[0].axvline(x=N_RUNS * 0.5, color='orange', linestyle='--', linewidth=2, label='50% threshold')
    axes[0].set_xlabel(f'Times in Blanket (out of {N_RUNS})')
    axes[0].set_title("Clinical Triage AI\nTarget: LENGTH_OF_STAY\n(ILP-Optimal DAG)", fontweight='bold')
    axes[0].legend()

if financial_counter:
    vars_f = sorted(financial_counter.keys(), key=lambda x: financial_counter[x], reverse=True)
    counts_f = [financial_counter[v] for v in vars_f]
    colors_f = ['#2ecc71' if c >= N_RUNS * 0.8 else '#f39c12' if c >= N_RUNS * 0.5 else '#e74c3c' for c in counts_f]
    axes[1].barh(vars_f, counts_f, color=colors_f, edgecolor='black')
    axes[1].axvline(x=N_RUNS * 0.8, color='green', linestyle='--', linewidth=2, label='80% threshold')
    axes[1].axvline(x=N_RUNS * 0.5, color='orange', linestyle='--', linewidth=2, label='50% threshold')
    axes[1].set_xlabel(f'Times in Blanket (out of {N_RUNS})')
    axes[1].set_title("Medical Billing AI\nTarget: TOTAL_CHARGES\n(ILP-Optimal DAG)", fontweight='bold')
    axes[1].legend()

plt.tight_layout()
plt.savefig("markov_blanket_stability.png", dpi=150, bbox_inches='tight')
plt.show()
print("Stability chart saved as 'markov_blanket_stability.png'")

# ==========================================
# 7. PREDICTION VALIDATION
# ==========================================
print("\n\n" + "=" * 65)
print("  PREDICTION VALIDATION (Random Forest, 5-fold CV)")
print("=" * 65)

df_pred = pd.read_csv("Cleaned_Patient_Vault.csv")
df_pred = df_pred.drop(columns=["RECORD_ID"])
df_pred = df_pred.sample(n=50000, random_state=42).copy()

df_pred['TOTAL_CHARGES'] = pd.qcut(df_pred['TOTAL_CHARGES'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
df_pred['LENGTH_OF_STAY'] = pd.qcut(df_pred['LENGTH_OF_STAY'], q=4, labels=['Short', 'Medium', 'Long', 'Extended'], duplicates='drop')

for col in df_pred.columns:
    le = LabelEncoder()
    df_pred[col] = le.fit_transform(df_pred[col].astype(str))

def compare_models(df, target, blanket_features, task_name):
    if not blanket_features:
        print(f"\n  {task_name}")
        print(f"  Target: {target}")
        print(f"  WARNING: No blanket features found — skipping comparison.")
        return None, None

    all_features = [c for c in df.columns if c != target]
    y = df[target]

    X_all = df[all_features]
    scores_all = cross_val_score(
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        X_all, y, cv=5, scoring='accuracy'
    )

    X_blanket = df[blanket_features]
    scores_blanket = cross_val_score(
        RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        X_blanket, y, cv=5, scoring='accuracy'
    )

    drop = scores_all.mean() - scores_blanket.mean()

    print(f"\n  {task_name}")
    print(f"  Target: {target}")
    print(f"  Blanket vars: {blanket_features}")
    print(f"  ---")
    print(f"  ALL features ({len(all_features)} vars):     {scores_all.mean():.4f} (+/- {scores_all.std():.4f})")
    print(f"  BLANKET only  ({len(blanket_features)} vars):     {scores_blanket.mean():.4f} (+/- {scores_blanket.std():.4f})")
    print(f"  Accuracy drop:                  {drop:.4f}")
    print(f"  Features eliminated:            {len(all_features) - len(blanket_features)}")

    if drop < 0.02:
        print(f"  >>> MINIMAL DROP — blanket is sufficient for data minimization")
    elif drop < 0.05:
        print(f"  >>> SMALL DROP — blanket captures most of the signal")
    else:
        print(f"  >>> NOTABLE DROP — consider adding borderline (50%+) variables")

    return scores_all.mean(), scores_blanket.mean()

print("\nRunning 5-fold cross-validation...\n")

all_acc_c, blanket_acc_c = compare_models(df_pred, "LENGTH_OF_STAY", clinical_blanket, "Clinical Triage AI")
all_acc_f, blanket_acc_f = compare_models(df_pred, "TOTAL_CHARGES", financial_blanket, "Medical Billing AI")

# ==========================================
# 8. FINAL SUMMARY
# ==========================================
print("\n\n" + "=" * 65)
print("  FINAL DATA MINIMIZATION SUMMARY")
print(f"  DAG Learning Method: ILP (Globally Optimal BIC)")
print("=" * 65)
print(f"\n  Clinical AI (LENGTH_OF_STAY):")
print(f"    Keep: {clinical_blanket}")
if blanket_acc_c is not None:
    print(f"    Accuracy: {blanket_acc_c:.4f} vs {all_acc_c:.4f} (all features)")
else:
    print(f"    Accuracy: N/A (empty blanket)")

print(f"\n  Financial AI (TOTAL_CHARGES):")
print(f"    Keep: {financial_blanket}")
if blanket_acc_f is not None:
    print(f"    Accuracy: {blanket_acc_f:.4f} vs {all_acc_f:.4f} (all features)")
else:
    print(f"    Accuracy: N/A (empty blanket)")

print(f"\n  Conclusion: Collect ONLY the blanket variables.")
print(f"  Everything else can be dropped with minimal accuracy loss.")
print(f"  (DAG structure is provably optimal under BIC — not a local optimum)")
print("=" * 65)

# ==========================================
# 9. MARKOV BLANKET NETWORK GRAPHS
# ==========================================
print("\n\nGenerating Markov Blanket network diagrams...")
print("Running one final ILP structure learning on a large sample for the graph edges...")

df_graph = df_vault.sample(n=min(100000, len(df_vault)), random_state=99).copy()
df_graph['TOTAL_CHARGES'] = pd.qcut(df_graph['TOTAL_CHARGES'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
df_graph['LENGTH_OF_STAY'] = pd.qcut(df_graph['LENGTH_OF_STAY'], q=4, labels=['Short', 'Medium', 'Long', 'Extended'], duplicates='drop')
df_graph = df_graph.astype(str)

# Use ILP for the final graph too
final_edges = learn_dag_ilp(df_graph, max_parents=3, verbose=True)

G_full = nx.DiGraph()
G_full.add_nodes_from(df_graph.columns)
G_full.add_edges_from(final_edges)

targets = {
    "Task A: Clinical Triage AI": ("LENGTH_OF_STAY", clinical_blanket),
    "Task B: Medical Billing AI": ("TOTAL_CHARGES", financial_blanket)
}

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

for ax, (title, (target_node, mb_nodes)) in zip(axes, targets.items()):
    focus_nodes = [target_node] + mb_nodes
    sub_G = G_full.subgraph(focus_nodes).copy()

    for node in mb_nodes:
        if node not in sub_G.nodes():
            sub_G.add_node(node)

    pos = nx.spring_layout(sub_G, seed=42, k=2)

    node_colors = []
    for node in sub_G.nodes():
        if node == target_node:
            node_colors.append("#e74c3c")
        else:
            node_colors.append("#2ecc71")

    nx.draw_networkx_nodes(
        sub_G, pos, ax=ax,
        node_size=3500,
        node_color=node_colors,
        edgecolors="black",
        linewidths=2
    )

    nx.draw_networkx_edges(
        sub_G, pos, ax=ax,
        arrows=True,
        arrowstyle="-|>",
        arrowsize=25,
        edge_color="black",
        width=2,
        node_size=3500
    )

    nx.draw_networkx_labels(sub_G, pos, ax=ax, font_size=9, font_weight="bold")

    ax.axis("off")
    ax.set_title(f"{title}\nTarget: '{target_node}'\nStable Blanket ({len(mb_nodes)} vars)\n[ILP-Optimal DAG]",
                 fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("markov_blanket_graphs.png", dpi=150, bbox_inches='tight')
plt.show()
print("Network graphs saved as 'markov_blanket_graphs.png'")

Loading the full Patient Vault...
Total rows available: 706472

--- Running 1 bootstrap iterations with ILP-based DAG learning ---

Run 1/1...

Learning optimal DAG for 14 variables...
  Max parents per node: 3
  Sample size: 706000
  Pre-computing BIC scores for all candidate parent sets...
